# A chat app interface
A chat app with FastHTML and claudette/lisette

In [ ]:
#| default_exp chat

In [ ]:
#| hide
from IPython.display import HTML
from fasthtml.jupyter import *

In [ ]:
#| export
from fasthtml.common import *
from lisette import *

In [ ]:
#| export
from fastlite import *
import random
import string
import json
from dataclasses import dataclass, field

## Creating the app with daisy styling -

In [ ]:
#| export
app = FastHTML(hdrs=(
        Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
        Link(href='https://cdn.jsdelivr.net/npm/daisyui@5/themes.css', rel='stylesheet', type='text/css'),
        Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4'),
        Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/@tailwindcss/typography/dist/typography.min.css'),
        MarkdownJS(),
        ),
    )

In [ ]:
#| export
# For convenience
rt = app.route

In [ ]:
#| hide
#| eval: false
srv = JupyUvi(app)

## App State and DB

In [ ]:
#| export
#| eval: false
db = database('solveish.db')

In [ ]:
#| export

#class User: id:int
class Dialog: id:int; uid:int; messages:str; name:str

In [ ]:
#| export
#| eval: false
dialogs = db.create(Dialog, transform=True)

In [ ]:
#| export
@dataclass
class AppState:
    uid: int = 123
    h: list = field(default_factory=list)
    dlgid: int = field(default_factory=lambda: random.randint(1, 100000))
    dlgname: str = field(default_factory=lambda: ''.join(random.choices(string.ascii_lowercase, k=6)))
    model: str = 'claude-haiku-4-5'

    def randomize(self):
        self.h = []
        self.dlgid = random.randint(1, 100000)
        self.dlgname = ''.join(random.choices(string.ascii_lowercase, k=6))
        return self

    def setdlg(self, dlg):
        self.h = json.loads(dlg.messages)
        self.dlgid = dlg.id
        self.dlgname = dlg.name
    
    def add_msg(self, msg):
        self.h.append(msg)


In [ ]:
#| export
state = AppState()

## LLM system prompt -

In [ ]:
#| export
sysp = "You are a chat model, in a chat with a user. Given the chat history as context, answer the user's current question concisely."

## Chat with llm

In [ ]:
#| export
#| hide
class MsgType(enum.StrEnum):
    note = 'Note'
    prompt = 'Prompt'
    ai = 'AI'

In [ ]:
#| export
def bubble(msg:str, msg_type:MsgType=MsgType.prompt):
    if msg_type == MsgType.ai:
        return Div(Div(msg, cls='chat-bubble chat-bubble-secondary marked prose'), cls='chat chat-end')
    elif msg_type == MsgType.prompt:
        return Div(Div(msg, cls='chat-bubble chat-bubble-primary'), cls='chat chat-start')
    else:
        return Div(Div(msg, cls='chat-bubble chat-bubble-warning marked prose'), cls='chat chat-start')

In [ ]:
content = """
Here are some _markdown_ elements.

- This is a list item
- This is another list item
- And this is a third list item

**Fenced code blocks work here.**

## A title
Here is some text!
### Another title

"""

In [ ]:
#bubble(content, MsgType.note)

In [ ]:
#bubble("Can you help?")

In [ ]:
#| export
def mk_prompt(msg, h):
    return "Past messages: " + json.dumps(h) + "\n\nCurrent question: " + msg

In [ ]:
mk_prompt("Hello", ["Old message"])

'Past messages: ["Old message"]\n\nCurrent question: Hello'

In [ ]:
#| export
models = ['claude-haiku-4-5', 'claude-opus-4-5', 'claude-sonnet-4-5']
models

['claude-haiku-4-5', 'claude-opus-4-5', 'claude-sonnet-4-5']

In [ ]:
#| export
@rt
def setmodel(name:str):
    state.model = name
    return Summary(state.model, cls='btn m-1', id="selmodel")

In [ ]:
#| export
def modeldropdown():
    return Details(cls='dropdown dropdown-top')(
        Summary(state.model, cls='btn m-1', id="selmodel"),
        Ul(cls='menu dropdown-content bg-base-100 rounded-box z-1 w-52 p-2 shadow-sm')(
            (Li(A(m, hx_post=setmodel.to(name=m), hx_target='#selmodel', hx_swap="outerHTML", hx_on_click="this.closest('details').removeAttribute('open')")) for m in models)
        )
    )

In [ ]:
#| export
def llm_rsp(msg):
    return Div(
        Span(cls='loading loading-ring loading-md'), cls='flex justify-center',
        hx_post=ask_llm, hx_trigger='load', hx_swap='outerHTML',
        hx_vals=f'{{"msg": "{msg}"}}'
        )

In [ ]:
#| export
js_ui = '''
document.querySelectorAll('input[name="msg_type"]').forEach(radio => {
    radio.addEventListener('change', () => {
        let input = document.getElementById('msg');
        input.placeholder = radio.value === 'note' ? 'Markdown...' : 'Ask LLM...';
        input.className = radio.value === 'note' ? 'input input-warning' : 'input input-primary';

        let button = document.getElementById('send');
        button.textContent = radio.value === 'note' ? 'Add Note' : 'Prompt LLM';
        button.className = radio.value === 'note' ? 'btn btn-warning' : 'btn btn-primary';
    });
});
'''

In [ ]:
#| export
js_submit = '''
document.getElementById('msg').addEventListener('keydown', (e) => {
    if ((e.shiftKey) && e.key === 'Enter') {
        e.preventDefault();
        document.getElementById('send').click()
    }
});
'''

In [ ]:
#| export
@rt
def ask_llm(msg:str):
    r = Chat(model=state.model, sp=sysp)(mk_prompt(msg, state.h))
    rsp = r.choices[0].message.content
    # Add both user and llm response to history after calling model and receiving a response
    update_h(msg, MsgType.prompt)
    update_h(rsp, MsgType.ai)
    return bubble(rsp, MsgType.ai)

In [ ]:
#| export
@rt
def send(msg: str, msg_type:str):
    msg_type = MsgType[msg_type]
    if msg_type == MsgType.note:
        update_h(msg, msg_type)
        return bubble(msg, msg_type)
    else: return (bubble(msg, msg_type), llm_rsp(msg))

In [ ]:
#| export
def update_h(content:str, author:MsgType):
    state.add_msg({'content': content, 'author': str(author)}) 
    return dialogs.upsert(id=state.dlgid, uid=state.uid, messages=json.dumps(state.h), name=state.dlgname)

In [ ]:
#| export
@rt
def load_dialog(dlgid:int):
    dlg = dialogs[dlgid]
    state.setdlg(dlg)
    return (bubble(m['content'], MsgType(m['author']))for m in state.h)

In [ ]:
#| export
@rt
def dlgdropdown():
    udlgs = dialogs('uid=?', [123])

    return Details(cls='dropdown')(
        Summary('Dialogues', cls='btn m-1'),
        Ul(cls='menu dropdown-content bg-base-100 rounded-box z-1 w-52 p-2 shadow-sm')(
            (Li(A(dlg.name, hx_post=load_dialog.to(dlgid=dlg.id), hx_target='#chat', hx_swap='innerHTML')) for dlg in udlgs)
        )
    )

In [ ]:
#| export
@rt
def chatpg():
    state.randomize()
    return Div(
        Div(id='chat', style='width:100%; box-sizing:border-box; padding:0 1rem; border: 1px solid #ccc; border-radius: 0.5rem; min-height:400px; height: calc(100vh - 58px); overflow-y: auto;')(
        ),
        
        Form(
            Div(
                Label(Input(type='radio', name='msg_type', value='prompt', checked='checked', cls='radio radio-primary'), 'Prompt', style="padding: 1rem;"),
                Label(Input(type='radio', name='msg_type', value='note', cls='radio radio-secondary'), 'Note', style="padding: 1rem;"),
                Button('Prompt LLM', id='send', hx_post=send, hx_target='#chat', hx_swap='beforeend', cls='btn btn-primary', style="padding: 1rem;"),
                modeldropdown(),
                dlgdropdown(),
                style="padding: 1rem; display:flex; justify-content:center; align-items:center"
            ),
            Textarea(id='msg', type='text', placeholder='Type something...', style='width: 100%;', cls='input input-primary'),
            
            style='position: fixed; bottom: 0; width: 100%; padding:1rem'
        ),
        Script(js_ui),
        Script(js_submit),
        style='width: 100%'
    )

In [ ]:
#| hide
#| eval: false
srv.stop()

In [ ]:
#| hide
from nbdev import nbdev_export; nbdev_export()